#Mount Drive & verify GPU


In [1]:
# Cell 1 — Mount Drive & verify GPU
from google.colab import drive
import os

drive.mount('/content/drive')
gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print('Drive mounted.')
print('GPU:', gpu if gpu else '❌ NOT FOUND')

Mounted at /content/drive
Drive mounted.
GPU: Tesla T4


#Clone Repo & Setup


In [2]:
# Cell 2 — Clone repo & setup
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'Pillow', 'numpy',
                'torch', 'torchvision'], check=True)
print('✅ Packages installed')

REPO = '/content/project'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone',
        'https://github.com/Shayan19950405/FAMLDL.git',
        REPO], check=True)
print('✅ Repo ready')

✅ Packages installed
✅ Repo ready


# Download & Extract Datasets

In [3]:
# Cell 3 — Download & extract datasets
import subprocess, os, zipfile

DATASET_DIR = '/content/anomaly_datasets'
ZIP_PATH = '/content/drive/MyDrive/FAMLDL/Anomaly_Validation_Datasets.zip'

os.makedirs(DATASET_DIR, exist_ok=True)

if not os.path.exists(DATASET_DIR + '/Validation_Dataset'):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
print('✅ Datasets ready')

✅ Datasets ready


#Copy Checkpoints

In [4]:
# Cell 4 — Copy checkpoints
import shutil, os

os.makedirs('/content/checkpoints', exist_ok=True)

for fname in ['eomt_coco.bin', 'eomt_cityscapes.bin']:
    src = f'/content/drive/MyDrive/FAMLDL/{fname}'
    dst = f'/content/checkpoints/{fname}'
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ Copied {fname}')
    else:
        print(f'✅ {fname} ready')

print('✅ All checkpoints ready')

✅ Copied eomt_coco.bin
✅ Copied eomt_cityscapes.bin
✅ All checkpoints ready


# All Functions

In [5]:
# Cell 5 — All Functions
import os, sys, glob, json
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score, roc_curve
from tqdm import tqdm

os.chdir('/content/project/eomt')
sys.path.insert(0, '/content/project/eomt')
from models.eomt import EoMT
from models.vit import ViT

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ANOMALY_ROOT = Path('/content/anomaly_datasets/Validation_Dataset')
DATASETS = {
    'SMIYC_RA21':   ANOMALY_ROOT / 'RoadAnomaly21/images/*.png',
    'SMIYC_RO21':   ANOMALY_ROOT / 'RoadObsticle21/images/*.webp',
    'FS_LostFound': ANOMALY_ROOT / 'FS_LostFound_full/images/*.png',
    'FS_Static':    ANOMALY_ROOT / 'fs_static/images/*.jpg',
    'RoadAnomaly':  ANOMALY_ROOT / 'RoadAnomaly/images/*.jpg',
}

def load_eomt(ckpt_path, num_classes, num_q, img_size):
    enc = ViT(img_size=img_size, patch_size=16,
              backbone_name='vit_base_patch14_reg4_dinov2')
    model = EoMT(encoder=enc, num_classes=num_classes, num_q=num_q,
                 num_blocks=3, masked_attn_enabled=True)
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    clean = {k[len('network.'):]: v for k, v in sd.items()
             if k.startswith('network.')}
    model.load_state_dict(clean, strict=False)
    return model.eval().to(DEVICE)

@torch.no_grad()
def to_per_pixel_logits(mask_logits, class_logits):
    cls = class_logits.softmax(dim=-1)[..., :-1]
    return torch.einsum('bqhw,bqc->bchw', mask_logits.sigmoid(), cls)

def score_msp(pl):
    return (1 - pl.softmax(dim=0).max(dim=0).values).cpu().numpy()

def score_maxlogit(pl):
    return (-pl.max(dim=0).values).cpu().numpy()

def score_maxentropy(pl):
    p = pl.softmax(dim=0)
    return (-(p * p.log()).sum(dim=0)).cpu().numpy()

def score_rba(pl):
    # RbA: Nayal et al. ECCV 2022 — sum over classes
    return (-pl.sum(dim=0)).cpu().numpy()

def fpr_at_95_tpr(scores, labels):
    fpr, tpr, _ = roc_curve(labels, scores)
    return fpr[np.argmin(np.abs(tpr - 0.95))]

def compute_metrics(score_list, gt_list):
    scores = np.concatenate([s.flatten() for s in score_list])
    gts    = np.concatenate([g.flatten() for g in gt_list])
    valid  = gts != 255
    scores = scores[valid]
    gts    = gts[valid]
    ood = scores[gts == 1]
    ind = scores[gts == 0]
    val_out   = np.concatenate([ind, ood])
    val_label = np.concatenate([np.zeros(len(ind)), np.ones(len(ood))])
    auprc = average_precision_score(val_label, val_out) * 100
    fpr   = fpr_at_95_tpr(val_out, val_label) * 100
    return auprc, fpr

def load_gt(path, pathGT, img_size):
    mask = np.array(
        Image.open(pathGT).resize(
            (img_size[1], img_size[0]), Image.NEAREST))
    if 'RoadAnomaly' in pathGT:
        mask = np.where(mask == 2, 1, mask)
    return mask

def eval_one(model, glob_pattern, img_size):
    paths = sorted(glob.glob(str(glob_pattern)))
    if not paths:
        print(f'    [WARN] No images found: {glob_pattern}')
        return None
    input_tf = Compose([Resize(img_size, Image.BILINEAR), ToTensor()])
    scores = {m: [] for m in ['msp','maxlogit','maxentropy','rba']}
    gts = []
    for path in tqdm(paths, leave=False):
        pathGT = path.replace('images', 'labels_masks')
        if 'RoadObsticle21' in pathGT: pathGT = pathGT.replace('webp','png')
        if 'fs_static'      in pathGT: pathGT = pathGT.replace('jpg','png')
        if 'RoadAnomaly'    in pathGT: pathGT = pathGT.replace('jpg','png')
        if not os.path.exists(pathGT): continue
        gt = load_gt(path, pathGT, img_size)
        if 1 not in np.unique(gt): continue
        img = input_tf(Image.open(path).convert('RGB')).to(DEVICE)
        with torch.no_grad():
            ml, cl = model(img.unsqueeze(0) / 255.0)
            ml = F.interpolate(ml[-1], img_size,
                               mode='bilinear', align_corners=False)
            cl = cl[-1]
            pl = to_per_pixel_logits(ml, cl).squeeze(0)
        scores['msp'].append(score_msp(pl))
        scores['maxlogit'].append(score_maxlogit(pl))
        scores['maxentropy'].append(score_maxentropy(pl))
        scores['rba'].append(score_rba(pl))
        gts.append(gt)
        torch.cuda.empty_cache()
    if not gts:
        print('    [WARN] No valid images found')
        return None
    results = {}
    for m, sl in scores.items():
        auprc, fpr = compute_metrics(sl, gts)
        results[m] = {'auprc': round(auprc,2), 'fpr95': round(fpr,2)}
        print(f'    {m:<12} AuPRC={auprc:.1f}  FPR95={fpr:.1f}')
    return results

print('✅ All functions defined!')

✅ All functions defined!


# Main Evaluation (3 models × 5 datasets)


In [6]:
# Cell 6 — Main Evaluation (3 models × 5 datasets)
CHECKPOINTS = {
    'EoMT_COCO': {
        'path': '/content/checkpoints/eomt_coco.bin',
        'num_classes': 133, 'num_q': 200, 'img_size': (640, 640),
    },
    'EoMT_Cityscapes': {
        'path': '/content/checkpoints/eomt_cityscapes.bin',
        'num_classes': 19, 'num_q': 100, 'img_size': (1024, 1024),
    },
    'EoMT_Finetuned': {
        'path': '/content/drive/MyDrive/FAMLDL/checkpoints/exp_5_2_partial.bin',
        'num_classes': 19, 'num_q': 100, 'img_size': (1024, 1024),
    },
}

OUT = Path('/content/drive/MyDrive/FAMLDL/results/step8')
OUT.mkdir(parents=True, exist_ok=True)
all_results = {}

for ckpt_name, cfg in CHECKPOINTS.items():
    print(f'\n{"="*50}')
    print(f'Model: {ckpt_name}')
    print(f'{"="*50}')
    if not Path(cfg['path']).exists():
        print(f'[SKIP] checkpoint not found: {cfg["path"]}')
        continue
    model = load_eomt(cfg['path'], cfg['num_classes'],
                      cfg['num_q'], cfg['img_size'])
    all_results[ckpt_name] = {}
    for ds_name, ds_glob in DATASETS.items():
        print(f'\n  Dataset: {ds_name}')
        res = eval_one(model, ds_glob, cfg['img_size'])
        if res:
            all_results[ckpt_name][ds_name] = res
    with open(OUT / f'{ckpt_name}.json', 'w') as f:
        json.dump(all_results[ckpt_name], f, indent=2)
    print(f'\n✅ {ckpt_name} results saved')
    del model
    torch.cuda.empty_cache()

with open(OUT / 'all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('\n🎉 ALL EXPERIMENTS COMPLETE!')


Model: EoMT_COCO


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]


  Dataset: SMIYC_RA21


    msp          AuPRC=15.4  FPR95=94.6
    maxlogit     AuPRC=15.8  FPR95=84.2
    maxentropy   AuPRC=16.4  FPR95=92.8
    rba          AuPRC=15.6  FPR95=84.2

  Dataset: SMIYC_RO21


...

    msp          AuPRC=15.6  FPR95=93.1
    maxlogit     AuPRC=15.6  FPR95=93.0
    maxentropy   AuPRC=14.7  FPR95=93.9
    rba          AuPRC=15.6  FPR95=93.0

  Dataset: SMIYC_RO21


    msp          AuPRC=0.8  FPR95=90.8
    maxlogit     AuPRC=0.8  FPR95=90.7
    maxentropy   AuPRC=0.7  FPR95=93.7
    rba          AuPRC=0.8  FPR95=90.7

  Dataset: FS_LostFound


    msp          AuPRC=0.3  FPR95=93.9
    maxlogit     AuPRC=0.3  FPR95=93.8
    maxentropy   AuPRC=0.3  FPR95=94.0
    rba          AuPRC=0.3  FPR95=93.9

  Dataset: FS_Static


    msp          AuPRC=2.2  FPR95=92.3
    maxlogit     AuPRC=2.3  FPR95=92.6
    maxentropy   AuPRC=2.1  FPR95=93.9
    rba          AuPRC=2.3  FPR95=92.6

  Dataset: RoadAnomaly


    msp          AuPRC=10.5  FPR95=92.6
    maxlogit     AuPRC=10.6  FPR95=92.5
    maxentropy   AuPRC=9.8  FPR95=93.9
    rba          AuPRC=10.6  FPR95=92.5

✅ EoMT_Finetuned results saved

🎉 ALL EXPERIMENTS COMPLETE!


In [7]:
# Cell 7 — Temperature Scaling
from scipy.special import softmax as sp_softmax
from scipy.optimize import minimize_scalar as _ms

input_tf = Compose([Resize((1024,1024), Image.BILINEAR), ToTensor()])
OUT = Path('/content/drive/MyDrive/FAMLDL/results/step8')

print('Loading EoMT_Finetuned...')
model = load_eomt(
    '/content/drive/MyDrive/FAMLDL/checkpoints/exp_5_2_partial.bin',
    num_classes=19, num_q=100, img_size=(1024, 1024)
)
print('✅ Model loaded')

print('\nCollecting logits for calibration...')
all_logits, all_gts = [], []
for path in sorted(glob.glob(str(ANOMALY_ROOT / 'RoadAnomaly/images/*.jpg'))):
    pathGT = path.replace('images','labels_masks').replace('jpg','png')
    if not Path(pathGT).exists(): continue
    gt = load_gt(path, pathGT, (1024,1024))
    if 1 not in np.unique(gt): continue
    img = input_tf(Image.open(path).convert('RGB')).to(DEVICE)
    with torch.no_grad():
        ml, cl = model(img.unsqueeze(0) / 255.0)
        ml = F.interpolate(ml[-1], (1024,1024), mode='bilinear', align_corners=False)
        pl = to_per_pixel_logits(ml, cl[-1]).squeeze(0)
    all_logits.append(pl.cpu().numpy())
    all_gts.append(gt)
    torch.cuda.empty_cache()
print(f'Collected {len(all_logits)} images')

def nll(T):
    T = max(T, 1e-3)
    total, count = 0, 0
    for logits, gt in zip(all_logits, all_gts):
        valid = gt != 255
        flat = logits[:, valid].T
        normal = gt[valid] == 0
        if normal.sum() == 0: continue
        scaled = flat[normal] / T
        probs = sp_softmax(scaled, axis=1)
        total += -np.log(probs.max(axis=1) + 1e-8).mean()
        count += 1
    return total / max(count, 1)

result = _ms(nll, bounds=(0.01, 10.0), method='bounded')
T_opt = result.x
print(f'\n✅ Optimal T = {T_opt:.4f}')

TEMPS  = [1.0, 0.5, 0.75, 1.1, T_opt]
LABELS = ['MSP', 'MSP (T=0.5)', 'MSP (T=0.75)', 'MSP (T=1.1)', 'MSP (best T)']
table2 = {}

for T, label in zip(TEMPS, LABELS):
    print(f'\n── {label} (T={T:.3f}) ──')
    table2[label] = {'T': float(T)}
    for ds_name, ds_glob in DATASETS.items():
        paths = sorted(glob.glob(str(ds_glob)))
        score_list, gt_list = [], []
        for path in paths:
            pathGT = path.replace('images','labels_masks')
            if 'RoadObsticle21' in pathGT: pathGT = pathGT.replace('webp','png')
            if 'fs_static'      in pathGT: pathGT = pathGT.replace('jpg','png')
            if 'RoadAnomaly'    in pathGT: pathGT = pathGT.replace('jpg','png')
            if not Path(pathGT).exists(): continue
            gt = load_gt(path, pathGT, (1024,1024))
            if 1 not in np.unique(gt): continue
            img = input_tf(Image.open(path).convert('RGB')).to(DEVICE)
            with torch.no_grad():
                ml, cl = model(img.unsqueeze(0) / 255.0)
                ml = F.interpolate(ml[-1], (1024,1024), mode='bilinear', align_corners=False)
                pl = to_per_pixel_logits(ml, cl[-1]).squeeze(0)
            score_list.append(score_msp(pl / T))
            gt_list.append(gt)
            torch.cuda.empty_cache()
        if not gt_list: continue
        auprc, fpr = compute_metrics(score_list, gt_list)
        table2[label][ds_name] = {'auprc': round(auprc,2), 'fpr95': round(fpr,2)}
        print(f'  {ds_name}: AuPRC={auprc:.1f} FPR95={fpr:.1f}')

with open(OUT / 'table2_temperature_scaling.json', 'w') as f:
    json.dump(table2, f, indent=2)
print(f'\n✅ Temperature scaling done! (best T={T_opt:.4f})')
del model
torch.cuda.empty_cache()

Loading EoMT_Finetuned...
✅ Model loaded

Collected 60 images

✅ Optimal T = 0.0100

── MSP (T=1.000) ──
  SMIYC_RA21: AuPRC=15.6 FPR95=93.1
  SMIYC_RO21: AuPRC=0.8 FPR95=90.8
  FS_LostFound: AuPRC=0.3 FPR95=93.9
  FS_Static: AuPRC=2.2 FPR95=92.3
  RoadAnomaly: AuPRC=10.5 FPR95=92.6

── MSP (T=0.5) (T=0.500) ──
  SMIYC_RA21: AuPRC=15.7 FPR95=92.9
  SMIYC_RO21: AuPRC=0.8 FPR95=90.5
  FS_LostFound: AuPRC=0.3 FPR95=93.8
  FS_Static: AuPRC=2.3 FPR95=92.6
  RoadAnomaly: AuPRC=10.5 FPR95=92.5

── MSP (T=0.75) (T=0.750) ──
  SMIYC_RA21: AuPRC=15.6 FPR95=92.9
  SMIYC_RO21: AuPRC=0.8 FPR95=90.8
  FS_LostFound: AuPRC=0.3 FPR95=93.8
  FS_Static: AuPRC=2.3 FPR95=92.8
  RoadAnomaly: AuPRC=10.5 FPR95=92.5

── MSP (T=1.1) (T=1.100) ──
  SMIYC_RA21: AuPRC=15.5 FPR95=93.0
  SMIYC_RO21: AuPRC=0.8 FPR95=90.5
  FS_LostFound: AuPRC=0.3 FPR95=94.0
  FS_Static: AuPRC=2.2 FPR95=92.4
  RoadAnomaly: AuPRC=10.4 FPR95=92.3

── MSP (best T) (T=0.010) ──
  SMIYC_RA21: AuPRC=15.8 FPR95=93.0
  SMIYC_RO21: AuPRC=0.8 F